In [ ]:
"""
VECM Combination Search
-----------------------
Finds the best combination of metrics (from all possible subsets)
that minimizes MAPE for each individual metric using VECM.

Train : 2019-10-31 to 2024-12-31
Predict: 2025-01-31 to 2025-12-31
"""

import pandas as pd
import numpy as np
import warnings
from itertools import combinations

from statsmodels.tsa.vector_ar.vecm import VECM, select_order, select_coint_rank

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1.  LOAD YOUR DATA
#     Replace this block with your actual dataframe
# ─────────────────────────────────────────────
# Example structure (replace with your real df):
#   df columns → DATE, METRIC_A, METRIC_B, METRIC_C, ...
#
# df = pd.read_csv("your_file.csv", parse_dates=["DATE"])

# ── DEMO DATA (remove when using real data) ──────────────────────────────────
np.random.seed(42)
dates = pd.date_range("2019-10-31", "2025-12-31", freq="ME")
n = len(dates)

# Simulated cointegrated-ish series
base = np.cumsum(np.random.randn(n))
df = pd.DataFrame({"DATE": dates})
metric_cols = ["METRIC_A", "METRIC_B", "METRIC_C", "METRIC_D", "METRIC_E", "METRIC_F"]
for i, col in enumerate(metric_cols):
    df[col] = base + i * 0.5 + np.random.randn(n) * 0.5
# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────
# 2.  CONFIGURATION  (edit these)
# ─────────────────────────────────────────────
DATE_COL    = "DATE"
METRIC_COLS = ["METRIC_A", "METRIC_B", "METRIC_C",
               "METRIC_D", "METRIC_E", "METRIC_F"]

TRAIN_END   = "2024-12-31"
TRAIN_START = "2019-10-31"
PRED_START  = "2025-01-31"
PRED_END    = "2025-12-31"

MAX_LAGS    = 6      # max lag order to consider in lag selection
MIN_COMBO   = 2      # minimum combo size (VECM needs ≥ 2 variables)
# ─────────────────────────────────────────────

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.set_index(DATE_COL).sort_index()

train_df = df.loc[TRAIN_START:TRAIN_END, METRIC_COLS]
test_df  = df.loc[PRED_START:PRED_END,   METRIC_COLS]
n_pred   = len(test_df)   # should be 12

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
print(f"Predicting  : {n_pred} steps\n")


# ─────────────────────────────────────────────
# 3.  HELPERS
# ─────────────────────────────────────────────
def mape(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Mean Absolute Percentage Error (ignores zeros in actual)."""
    actual    = np.array(actual,    dtype=float)
    predicted = np.array(predicted, dtype=float)
    mask = actual != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def fit_and_predict_vecm(train: pd.DataFrame, n_steps: int):
    """
    Fit VECM on `train` and return out-of-sample forecasts.

    Returns
    -------
    forecast_df : pd.DataFrame  shape (n_steps, n_cols)  or  None on failure
    coint_rank  : int
    lag_order   : int
    """
    try:
        # -- Lag order selection (AIC)
        lag_res   = select_order(train, maxlags=MAX_LAGS, deterministic="ci")
        lag_order = lag_res.aic if lag_res.aic and lag_res.aic > 0 else 1

        # -- Cointegration rank (Johansen trace, 5 % significance)
        coint_res   = select_coint_rank(train, det_order=0,
                                        k_ar_diff=lag_order, method="trace",
                                        signif=0.05)
        coint_rank  = coint_res.rank
        if coint_rank == 0:
            coint_rank = 1   # force rank=1 rather than skip

        # -- Fit VECM
        model    = VECM(train, k_ar_diff=lag_order,
                        coint_rank=coint_rank, deterministic="ci")
        result   = model.fit()

        # -- Forecast
        raw      = result.predict(steps=n_steps)   # ndarray (n_steps, k)
        forecast = pd.DataFrame(raw, columns=train.columns)

        return forecast, coint_rank, lag_order

    except Exception as e:
        return None, None, None


# ─────────────────────────────────────────────
# 4.  MAIN LOOP  — all combinations
# ─────────────────────────────────────────────
# best_result[metric] = {"mape": float, "combo": tuple, "rank": int, "lags": int}
best_result = {m: {"mape": np.inf, "combo": None, "rank": None, "lags": None}
               for m in METRIC_COLS}

all_records = []   # store every (combo, metric, mape) for later inspection

total_combos = sum(
    len(list(combinations(METRIC_COLS, r)))
    for r in range(MIN_COMBO, len(METRIC_COLS) + 1)
)
print(f"Total combinations to evaluate: {total_combos}\n")
print("-" * 60)

combo_count = 0
for size in range(MIN_COMBO, len(METRIC_COLS) + 1):
    for combo in combinations(METRIC_COLS, size):
        combo_count += 1
        combo_label = " + ".join(combo)

        train_sub = train_df[list(combo)]
        test_sub  = test_df[list(combo)]

        forecast, rank, lags = fit_and_predict_vecm(train_sub, n_pred)

        if forecast is None:
            print(f"[{combo_count:>3}/{total_combos}]  SKIP  {combo_label}")
            continue

        # Compute MAPE for every metric IN this combo
        for metric in combo:
            actual    = test_sub[metric].values
            predicted = forecast[metric].values
            m_val     = mape(actual, predicted)

            all_records.append({
                "combo"     : combo,
                "combo_size": size,
                "metric"    : metric,
                "mape"      : round(m_val, 4),
                "coint_rank": rank,
                "lag_order" : lags,
            })

            if m_val < best_result[metric]["mape"]:
                best_result[metric]["mape"]  = m_val
                best_result[metric]["combo"] = combo
                best_result[metric]["rank"]  = rank
                best_result[metric]["lags"]  = lags

        print(f"[{combo_count:>3}/{total_combos}]  OK    {combo_label}")

print("-" * 60)


# ─────────────────────────────────────────────
# 5.  RESULTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BEST COMBINATION PER METRIC")
print("=" * 60)
for metric, info in best_result.items():
    print(f"\n  Metric      : {metric}")
    print(f"  Best MAPE   : {info['mape']:.4f} %")
    print(f"  Combination : {list(info['combo'])}")
    print(f"  Coint. rank : {info['rank']}")
    print(f"  Lag order   : {info['lags']}")

# Save detailed results to CSV
results_df = pd.DataFrame(all_records)
results_df = results_df.sort_values(["metric", "mape"])
results_df["combo"] = results_df["combo"].apply(list)
results_df.to_csv("/mnt/user-data/outputs/vecm_all_combo_results.csv", index=False)

print("\n\nDetailed results saved to: vecm_all_combo_results.csv")

# ─────────────────────────────────────────────
# 6.  TOP-5 PER METRIC  (quick view)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  TOP 5 COMBOS PER METRIC")
print("=" * 60)
for metric in METRIC_COLS:
    sub = results_df[results_df["metric"] == metric].head(5)
    print(f"\n  {metric}")
    print(sub[["combo", "mape", "coint_rank", "lag_order"]].to_string(index=False))